# 気象条件にもとづく動画候補の抽出

気象データと、ファイル名に撮影日時を含む動画を照合して、条件に合う動画候補をCSVへ出力します。

- 動画のコピー・移動・削除は行いません。
- 上から順番にセルを実行してください。
- 出力は候補であり、目的の行動が写っているかは最終的に目視確認が必要です。


## 1. ライブラリを読み込む


In [ ]:
import re
from pathlib import Path
import pandas as pd


## 2. 設定を変更する

通常、変更するのはこのセルだけです。`r"..."` の中を、自分のPC上の場所に書き換えます。


In [ ]:
# --- ファイル・フォルダの場所 ---
WEATHER_CSV = Path(r"C:\path\to\weather.csv")
SUNRISE_SUNSET_FILES = [
    # (4, Path(r"C:\path\to\sunrise_sunset_04.csv")),
    # (5, Path(r"C:\path\to\sunrise_sunset_05.csv")),
]
VIDEO_ROOT = Path(r"C:\path\to\video_folder")
OUTPUT_CSV = Path(r"matched_videos.csv")

# --- 気象CSVの形式 ---
WEATHER_ENCODING = "cp932"
WEATHER_SKIP_ROWS = 6
WEATHER_COLUMNS = {"datetime": 0, "temperature": 1, "wind_speed": 4}
WEATHER_DATETIME_FORMAT = "%Y/%m/%d %H:%M:%S"

# --- 日の出・日没CSVの形式（国立天文台の月別CSVを想定） ---
YEAR = 2025
SUNRISE_SUNSET_ENCODING = "utf-8-sig"

# --- 抽出条件 ---
MIN_TEMPERATURE_C = 30.3       # 使わない場合は None
MAX_WIND_SPEED_M_S = 2.3       # 使わない場合は None
DAYTIME_ONLY = True
MATCH_TOLERANCE_MINUTES = 30
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov"}


## 3. 気象データを読み込む

件数が0なら、パス・文字コード・列番号を確認してください。


In [ ]:
if not WEATHER_CSV.is_file():
    raise FileNotFoundError(f"気象CSVが見つかりません: {WEATHER_CSV}")

raw_weather = pd.read_csv(WEATHER_CSV, encoding=WEATHER_ENCODING, header=None,
                          skiprows=WEATHER_SKIP_ROWS, dtype=str)
try:
    weather_df = pd.DataFrame({
        "weather_datetime": pd.to_datetime(raw_weather.iloc[:, WEATHER_COLUMNS["datetime"]].str.strip(),
                                             format=WEATHER_DATETIME_FORMAT, errors="coerce"),
        "temperature_c": pd.to_numeric(raw_weather.iloc[:, WEATHER_COLUMNS["temperature"]], errors="coerce"),
        "wind_speed_m_s": pd.to_numeric(raw_weather.iloc[:, WEATHER_COLUMNS["wind_speed"]], errors="coerce"),
    })
except IndexError as exc:
    raise ValueError("WEATHER_COLUMNS の列番号がCSVの列数を超えています。") from exc
weather_df = weather_df.dropna().sort_values("weather_datetime").reset_index(drop=True)
print(f"気象データ読み込み完了: {len(weather_df)}件")
weather_df.head()


## 4. 日の出・日没データを読み込む

`DAYTIME_ONLY = False` の場合は、このセルと次のセルは不要です。


In [ ]:
sun_records = []
for month, csv_path in SUNRISE_SUNSET_FILES:
    if not csv_path.is_file():
        raise FileNotFoundError(f"日の出・日没CSVが見つかりません: {csv_path}")
    table = pd.read_csv(csv_path, encoding=SUNRISE_SUNSET_ENCODING, header=None, skiprows=2, dtype=str)
    for _, row in table.iterrows():
        try:
            day = int(str(row.iloc[0]).strip())
            date = pd.Timestamp(year=YEAR, month=month, day=day)
            sun_records.append({
                "date": date.date(),
                "sunrise": pd.to_datetime(f"{date:%Y-%m-%d} {str(row.iloc[1]).strip()}", format="%Y-%m-%d %H:%M"),
                "sunset": pd.to_datetime(f"{date:%Y-%m-%d} {str(row.iloc[5]).strip()}", format="%Y-%m-%d %H:%M"),
            })
        except (IndexError, TypeError, ValueError):
            continue
sun_df = pd.DataFrame(sun_records).drop_duplicates("date")
print(f"日の出・日没データ読み込み完了: {len(sun_df)}日分")
sun_df.head()


## 5. 昼間データと気象条件で絞り込む


In [ ]:
if DAYTIME_ONLY:
    if sun_df.empty:
        raise ValueError("日の出・日没データがありません。設定を確認してください。")
    lookup = sun_df.set_index("date")
    filtered_weather = weather_df.copy()
    dates = filtered_weather["weather_datetime"].dt.date
    filtered_weather["sunrise"] = dates.map(lookup["sunrise"])
    filtered_weather["sunset"] = dates.map(lookup["sunset"])
    filtered_weather = filtered_weather.loc[
        filtered_weather["sunrise"].notna()
        & filtered_weather["weather_datetime"].between(filtered_weather["sunrise"], filtered_weather["sunset"])
    ].drop(columns=["sunrise", "sunset"])
else:
    filtered_weather = weather_df.copy()

if MIN_TEMPERATURE_C is not None:
    filtered_weather = filtered_weather[filtered_weather["temperature_c"] >= MIN_TEMPERATURE_C]
if MAX_WIND_SPEED_M_S is not None:
    filtered_weather = filtered_weather[filtered_weather["wind_speed_m_s"] <= MAX_WIND_SPEED_M_S]
filtered_weather = filtered_weather.reset_index(drop=True)
print(f"条件に合う気象時刻: {len(filtered_weather)}件")
filtered_weather


## 6. 動画フォルダを一度だけ走査する

`YYYYMMDD_HHMMSS`、ハイフン区切り、半角スペース区切り、連結形式の日時をファイル名から取得します。


In [ ]:
if not VIDEO_ROOT.is_dir():
    raise NotADirectoryError(f"動画フォルダが見つかりません: {VIDEO_ROOT}")

timestamp_pattern = re.compile(r"(?<!\d)(?P<date>\d{8})[ _-]?(?P<time>\d{6})(?!\d)")
video_records = []
for path in VIDEO_ROOT.rglob("*"):
    if not (path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS):
        continue
    match = timestamp_pattern.search(path.name)
    if match is None:
        continue
    video_datetime = pd.to_datetime(match.group("date") + match.group("time"),
                                    format="%Y%m%d%H%M%S", errors="coerce")
    if pd.notna(video_datetime):
        video_records.append({"video_path": path.relative_to(VIDEO_ROOT).as_posix(),
                              "filename": path.name, "video_datetime": video_datetime})
videos_df = pd.DataFrame(video_records, columns=["video_path", "filename", "video_datetime"])
videos_df = videos_df.sort_values("video_datetime").reset_index(drop=True)
print(f"日時を取得できた動画: {len(videos_df)}本")
videos_df.head()


## 7. 動画と気象時刻を照合する

各動画には、条件を満たす気象時刻のうち最も近いものを対応付けます。


In [ ]:
matched_df = pd.merge_asof(
    videos_df.sort_values("video_datetime"),
    filtered_weather.sort_values("weather_datetime"),
    left_on="video_datetime", right_on="weather_datetime",
    direction="nearest", tolerance=pd.Timedelta(minutes=MATCH_TOLERANCE_MINUTES),
).dropna(subset=["weather_datetime"])

matched_df["time_difference_minutes"] = (
    (matched_df["video_datetime"] - matched_df["weather_datetime"]).abs().dt.total_seconds() / 60
)
matched_df = matched_df[["video_path", "filename", "video_datetime", "weather_datetime",
                         "temperature_c", "wind_speed_m_s", "time_difference_minutes"]]
print(f"照合できた動画: {len(matched_df)}本")
matched_df.head()


## 8. CSVとして保存する

`video_path` は動画フォルダからの相対パスです。動画ファイル自体には変更を加えません。


In [ ]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
matched_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"完了: {OUTPUT_CSV.resolve()}")
print(f"出力した動画候補: {len(matched_df)}本")
